# Staged earth-fault test – evaluate one measurement

Evaluation of one COMTRADE recording of a staged earth-fault test in a compensated (resonant-earthed) network. All
computations are in the package **`groundfault`** – the notebook is only the user interface. Reference copy:
`templates/evaluate_measurement_DEFAULT.ipynb`.

**Procedure**
1. **Configuration** – campaign, file, additional impedance; your limits, margins and factors (groundfault has none).
   Leave the analysis intervals empty at first.
2. **Open the file** – check the channel roles and the raw-data diagnostics.
3. **RMS trend** – read the times of the *steady-state fault*, *pre-fault* and *post-fault* intervals, enter them in
   section 1 and run the notebook again (*Run All*). Nothing is automatic: of all 200-ms windows in the steady-state
   interval the window with the **largest** residual current is assessed.
4. **Evaluation** – assessment quantities with your measurement-uncertainty margins, angle method, plausibility checks, voltages
   before/during/after the fault, station and transformer if measured. At a **station**: earth potential rise and
   touch voltage per frequency (direct and fault-related), PEN currents and current into the LV network (Z_E,LV),
   comparison with the **earthing measurement from groundmeas** and **reduction factor**.
5. **Signal explorer** – look at all channels: RMS value over time with harmonics/THD on hover, instantaneous values
   and spectrum of a freely chosen excerpt (check that the numbers match the curves).
6. **Checks** (optional) – mains frequency, spectra per window, comparison with the RMS values of the recorder,
   resonance curve.
7. **Export** – result file (input of `evaluate_campaign.ipynb`), tables (CSV/XLSX/DOCX/Markdown), figures
   (PDF/SVG/PNG, without titles, A4 type area) and the signal explorer as an HTML file; entry in groundmeas.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

LANGUAGE = "en"                # output language of tables, hints and figures: "en" | "de"
try:
    import groundfault as gf
except ImportError:            # not installed: use the source folder of a clone (…/groundfault/src)
    for p in [Path.cwd(), *Path.cwd().parents]:    # ./src, ./groundfault/src, ./*/groundfault/src
        try:
            candidates = [p / "src", p / "groundfault" / "src", *sorted(p.glob("*/groundfault/src"))]
        except OSError:
            candidates = [p / "src", p / "groundfault" / "src"]
        hits = [c for c in candidates if (c / "groundfault" / "__init__.py").is_file()]
        if hits:
            sys.path.insert(0, str(hits[0]))
            break
    import groundfault as gf
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from groundfault import (campaign, corrections, demo, diagnostics, earthing, equivalent_source, explorer, i18n,
                         interactive, layout, legacy, register, report, rms, sites)

i18n.set_language(LANGUAGE)
# work folder = this folder or the first folder above it with a campaigns folder ("campaigns" or
# "01_Messkampagnen"); its name gives the folder layout, without one the layout of LANGUAGE is used
WORK_FOLDER, LAYOUT = layout.find_work_folder(Path.cwd(), default=LANGUAGE)
WORK_FOLDER = Path(WORK_FOLDER)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: "%.4g" % x)
report.activate_style()        # notebook preview = typeface of the exported figures


def show(df):
    """Table with column names and codes in the output language."""
    display(i18n.localize(df))


print("groundfault %s · work folder: %s · layout: %s · font of the figures: %s"
      % (gf.__version__, WORK_FOLDER, LAYOUT.language, report.used_font()))

## 1 · Configuration

In [ ]:
# ---------------------------------------------------------------- measurement
CAMPAIGN    = "YYYY-MM-DD_Place"                     # folder in the campaigns folder
FILE        = "raw/<site>/<file>.cfg"                 # .cfg relative to the campaign folder
MEASUREMENT = None                                    # name for exports (None = file name without extension)
Z_FAULT     = 0.0                                     # additional impedance in the fault branch in Ω (substation: 0 / 2.5 / 5, station: None)
DEMO        = False                                   # True: create the demo campaign if it does not exist yet

# ---------------------------------------------------------------- analysis intervals in s (read them in section 3!)
STEADY_STATE = None   # (t0, t1) steady-state fault – after the transient, before an active residual current increase
PRE_FAULT    = None   # (t0, t1) operation without fault before the fault
POST_FAULT   = None   # (t0, t1) after the decay of the neutral displacement voltage

# ---------------------------------------------------------------- limits, margins and factors – defined by you
# groundfault contains no limits or safety factors: take them from the operator's specification or the applicable
# standard. Not set (the values below), the parts that need them are skipped with a note.
LIMITS            = ()      # limits of the residual current in A, e.g. (I_1, I_2): extrapolation of the station voltages
U_TP              = None    # permissible touch voltage U_Tp in V (relevance of the reduction factor)
F_FACTOR          = None    # factor F of the permissible earth potential rise F·U_Tp
REFERENCE_CURRENT = None    # reference current in A for the reduction factor (None = the largest of LIMITS)
UNCERTAINTY_I     = 0.0     # measurement-uncertainty margin m_I of currents as a fraction: I·(1 + m_I) (0.0 = none)
UNCERTAINTY_U     = 0.0     # measurement-uncertainty margin m_U of voltages as a fraction: U·(1 − m_U) (0.0 = none)
PQ_SAFETY_FACTOR  = 1.0     # factor on the 95 % quantile of a PQ measurement (1.0 = none)
R_TYPICAL         = None    # range (low, high) of the reduction factor at 50 Hz you expect, e.g. from your experience
# (LIMITS, U_TP, F_FACTOR, UNCERTAINTY_I/_U and PQ_SAFETY_FACTOR become the settings ires_limits, u_tp, f_factor,
#  uncertainty_I/_U and pq_safety_factor and are stored in the result file)

# ---------------------------------------------------------------- evaluation parameters
SETTINGS = gf.Settings(
    T_win=0.200,                 # window length in s (step = window length)
    harmonics=(1, 3, 5, 7),      # orders of the assessment quantities
    uncertainty_I=UNCERTAINTY_I,  # increases currents
    uncertainty_U=UNCERTAINTY_U,  # decreases voltages
    u_thd95_percent=None,        # 95 % quantile of the THD (line-to-line) of a PQ measurement in % (None = no scaling)
    pq_safety_factor=PQ_SAFETY_FACTOR,
    correct_polarity=True,       # Iw < 0 → rotate the current phasors by 180° (clamp reversed)
    ires_limits=LIMITS, u_tp=U_TP, f_factor=F_FACTOR,
)

# ---------------------------------------------------------------- channels (only if needed)
# Permanent corrections of the campaign (wrong CT ratio, clamp around one of several conductors, polarity, channel
# to be ignored, main channel) belong in notes/corrections.csv of the campaign –
# they are applied automatically and documented in the result. Here only settings for this run:
APPLY_CORRECTIONS = True
ROLE_OVERRIDES = {}      # e.g. {"ISp22": None} = ignore the channel, {"U1": "u_l1e"} = fix the role
MAIN_CHANNEL   = {}      # e.g. {"i_res": "IF2"} (default: IF before IF1 before IFr with the largest range)
ALIGNMENT      = None    # time offset between subsystems, e.g. dict(roles=["u_l1e", "u_l2e", "u_l3e"], dt_s=1.5)
FS_OVERRIDE    = None    # force the sampling rate (only for a defective time base, see the diagnostics)

# ---------------------------------------------------------------- checks and export
RMS_FILE         = None      # RMS export of the recorder (check), e.g. "raw/<site>/<file>_10_000Hz.cfg"
V_CURVE          = False   # fundamental trend / resonance curve (coil adjustment during the fault)
SHOW_ENVELOPE    = False   # also the envelope of the instantaneous values of the whole recording (interactive)
EXPORT_FORMATS   = ("pdf", "svg", "png")
RECORD_INTERVALS = True    # document the chosen intervals in the register of the work folder (not with DEMO = True)

# ---------------------------------------------------------------- signal explorer and earthing measurement (station)
SIGNAL_EXPLORER  = True    # section 5: look at all channels (hover with RMS value, harmonics, THD)
EXPLORER_HTML    = True    # also export the explorer as an HTML file (opens in a browser without Python)
GROUNDMEAS_DB    = None   # None = as groundmeas (environment variable GROUNDMEAS_DB or ~/.config/groundmeas/config.json)
GROUNDMEAS_LOCATION = None       # location in groundmeas; None = from the sites file of the campaign or the folder name
GROUNDMEAS_MEASUREMENTS = None   # comparison: ids of the measurements in groundmeas, e.g. [3, 21]; None = all of the location
GROUNDMEAS_EXPORT = False        # True: enter as "staged_fault_test" (after a backup copy); "replace": renew the own entry
GROUNDMEAS_UNCERTAIN = "mark"    # uncertain fault-related values: "mark" (with a remark) | "omit"
GROUNDMEAS_UNITS = "suitable"    # "suitable": readable per quantity (mV, mA …; groundmeas ≥ 1.6.1) | "base": V and A
GROUNDMEAS_DESCRIPTION = ""      # optional text before the automatic description, e.g. "20 to 0.4 kV, compensated"
GROUNDMEAS_TIMEZONE = None       # time zone of the recorder clock if it is not UTC, e.g. "Europe/Berlin"
GROUNDMEAS_TIMESTAMP = None      # only for a wrongly set recorder clock: start of the recording in UTC "YYYY-MM-DD hh:mm:ss"

## 2 · Open the file, channel roles, raw-data diagnostics

Check: Are all roles detected correctly (line-to-earth voltages, fault current, coils, station)? Is the right
**main channel** of the fault current marked (✓)? Does the diagnosis report saturation, flat channels, a deviating
time base or a time offset between voltage and current channels? The corrections of the campaign are applied here
and shown as a table.

In [ ]:
CAMPAIGN_FOLDER = WORK_FOLDER / LAYOUT.campaigns / CAMPAIGN
if DEMO and not CAMPAIGN_FOLDER.exists():
    demo.make_campaign(str(WORK_FOLDER), CAMPAIGN, layout=LAYOUT.language)
CFG = CAMPAIGN_FOLDER / FILE
SITE = sites.site_from_path(FILE)
CORR = corrections.load(CAMPAIGN_FOLDER) if APPLY_CORRECTIONS else corrections.Corrections.empty()
rec = gf.open_recording(str(CFG), fs_override=FS_OVERRIDE)
corrections_log = CORR.apply(rec)        # CT factors per channel – contained in all values from here on
roles = gf.detect_roles(rec.channel_names, rec.units, {**CORR.role_overrides(rec), **ROLE_OVERRIDES},
                        {**CORR.main_channel(rec), **MAIN_CHANNEL}, CORR.distances(rec))
MEASUREMENT = MEASUREMENT or CFG.stem
if len(corrections_log):
    display(Markdown("**Corrections of the campaign** (contained in all following values)"))
    show(corrections_log)
for h in CORR.hints:
    print("⚠", h)
kind = "+".join(str(i18n.value_label("measurement_kind", k)) for k in roles.measurement_kind().split("+"))
print(rec, "| start (cfg):", rec.t_start, "| kind of measurement:", kind)
show(roles.table())
dg = diagnostics.diagnose(rec, roles)
print(dg.text())
show(dg.channels)

## 3 · RMS trend – read the analysis intervals

Moving true RMS value (20-ms windows, DC removed per window). Read the times with zoom and range slider:
**steady state** = after the transient and *before* an active residual current increase or coil adjustment;
**pre-fault** and **post-fault** = operation without fault (after the decay of the neutral displacement voltage).
Entered intervals are shaded.

In [ ]:
INTERVALS = gf.Intervals(STEADY_STATE, PRE_FAULT, POST_FAULT)
interactive.rms_values(rec, roles, intervals=INTERVALS).show()
if SHOW_ENVELOPE:
    interactive.overview(rec, roles, intervals=INTERVALS).show()

## 4 · Evaluation

In [ ]:
res = None
if STEADY_STATE is None:
    display(Markdown("**➜ First enter `STEADY_STATE` (and `PRE_FAULT`/`POST_FAULT`) in section 1 and run again.**"))
else:
    res = gf.evaluate(rec, INTERVALS, SETTINGS, roles, alignment=ALIGNMENT, name=MEASUREMENT, z_fault_ohm=Z_FAULT,
                      metadata=dict(campaign=CAMPAIGN, site=SITE))
    print(res.summary())
    display(Markdown("**Assessment quantities** – measured value and assessment value with the measurement-uncertainty margins of section 1"))
    show(res.table("assessment_quantities"))
    display(Markdown("**Plausibility checks**"))
    show(res.table("plausibility"))

### Voltages before, during and after the fault

In [ ]:
if res is not None and "voltage" in res.values:
    show(res.table("voltage_metrics"))
    interactive.voltage_spectrum(res).show()

### Fault current – harmonics per window and statistics over the steady-state interval

In [ ]:
if res is not None and res.roles.has("i_res"):
    interactive.window_spectrum(res).show()
    interactive.harmonics_time(res).show()
    if not res.table("coil_currents").empty:
        show(res.table("coil_currents"))
    show(res.table("statistics"))

### Station (earth potential rise, touch voltage, shield current) and transformer – if measured

In [ ]:
if res is not None:
    for name in ("station_metrics", "station_voltages", "station_currents", "transformer_power"):
        if name in res.tables and not res.tables[name].empty:
            display(Markdown("**%s**" % report.table_title(name)))
            show(res.tables[name])

### Station: earth potential rise and touch voltage per frequency

Per probe and harmonic (I_F ≥ 1 % of the fundamental): **direct** = voltage/fault current during the fault, as
measured; **fault-related** = only the part caused by the fault current. Before the fault there is often already a
voltage at the station (**background**, e.g. currents from the LV network) – it is subtracted with three methods:
time reference at the fault inception, time reference at the fault clearance and (if measured) reference to the
line-to-line voltage.

**How reliable is the fault-related part?** `method_spread_pct` shows how well the methods agree. `uncertainty_pct`
estimates how much the background can distort the result: a fluctuating background (null test in the fault-free
intervals) and – without a voltage measurement – the frequency reference through the fault current, each × background
/ fault-related part. If one of them exceeds 20 %, `uncertain` is True and the figures show the value as **⊗** –
typical at 150 Hz, where the background is often larger than the fault-related part. The **touch voltage** comes from
the 1-m probe (U1m, UE1 …) or – as an approximation – from the shortest probe up to 5 m (see the role table in
section 2). For the comparison with the earthing measurement the fault-related part counts.

**PEN currents and LV network** (if `IPEN_v`/`IPEN_n` are measured): the same superposition separates the load
unbalance of the PEN current (present before the fault) from the fault-related current into the LV network I_LV.
`IPEN_v` (sum of the LV feeders, before the transformer star point) and `IPEN_n` (to the main earthing busbar) must
carry the same fault-related current (Kirchhoff's current law – the current of the transformer star point does not
change). **Z_E,LV = ΔU_E/I_LV** per frequency and as an RMS value – for every earth-potential probe and for the
**probe maximum** (`channel` = `max(UE)`, column `probe`): per frequency the highest voltage of all probes against the
station earth, as RMS value the probe with the highest RMS value. Potentials spread differently; each probe measures
at most the full earth potential rise. **Parallel check** (`parallel_check`): the LV network is a branch of the total
earthing, so |Z_E,LV| ≥ |Z_E,fault| = |max(U_E)/I_F| (equivalent to |I_LV| ≤ |I_F|) at every frequency; with a shield
current also against the earth current I_E = I_F − I_S (`earth_current_check`). Violated only if the deviation exceeds
the measurement uncertainty. The comparison with the earthing measurement follows in the next section.

In [ ]:
if res is not None and not res.tables.get("station_frequency", pd.DataFrame()).empty:
    show(res.table("station_frequency"))
    if LIMITS:
        display(Markdown("**Extrapolation to the limits of the residual current** (proportional; fault-related; with background)"))
        show(res.table("station_extrapolation"))
    else:
        print("ℹ Extrapolation to the limits of the residual current skipped – LIMITS is not set (section 1).")
    interactive.station_frequency(res).show()
    if not res.tables.get("station_currents_frequency", pd.DataFrame()).empty:
        display(Markdown("**PEN and shield currents per frequency** – measured during the fault, background (load unbalance) and fault-related part (share = ΔI/I_F)"))
        show(res.table("station_currents_frequency"))
    if not res.tables.get("station_lv_network", pd.DataFrame()).empty:
        display(Markdown("**LV network: current I_LV and earthing impedance Z_E,LV = ΔU_E/I_LV** (per frequency and RMS value; parallel check: Z_E,LV ≥ max(U_E)/I_F, earth-current check: |I_LV| ≤ |I_E|)"))
        show(res.table("station_lv_network"))
        show(res.table("station_lv_network_eff"))
        interactive.lv_network(res.table("station_lv_network")).show()
elif res is not None:
    print("No earth potential rise or touch voltage measured – section not applicable.")

### Station: comparison with the earthing measurement (groundmeas)

The earthing measurements (current injection) are kept in a **groundmeas database**, which is only **read** here.
The location comes from the sites file of the campaign, otherwise from the folder name (typing errors are tolerated)
– if needed set `GROUNDMEAS_LOCATION` or `GROUNDMEAS_MEASUREMENTS` in section 1. Compared is the **voltage per ampere**
(V/A = Ω) at the same frequency: injection (probe at a comparable distance) ↔ fault-related part of the fault test.
For the earth potential rise at a remote probe the ratio is the effective **reduction factor** (return current through
cable shields). Add probe distances that are not in the channel name (`UEr`, `UE`) to the corrections file (column
`distance_m`). The entry of this measurement in groundmeas follows in section 7.

**Reduction factor** r = Z_E,fault/Z_E,injection (earth potential rise per ampere, magnitudes only): per harmonic and
as an RMS value over all harmonics (weighted with the fault current); for comparison the RMS value from the measured
RMS values (with background, earthing impedance at 50 Hz) and – if measured – the cross-check from the shield
current. Decisive is the **probe maximum** of the fault test (`max(UE)`: highest voltage of all probes) against the
farthest probe of the earthing measurement; in addition each probe against the probe at a comparable distance. If the
earthing measurement lacks a frequency, it is interpolated between the neighbouring frequencies (never extrapolated).
**Classification:** r at 50 Hz against the range you expect (`R_TYPICAL` in section 1, e.g. from your experience
with comparable installations – groundfault has no default range); r should fall with the frequency (inductive
coupling of the return conductors, f → ∞: r → 0). Much smaller values indicate too high a value of the earthing
measurement (methodological error of the fall-of-potential method) or further return paths.
**Relevance for the assessment:** for small impedances relative deviations look large – the voltage decides. Both
impedances are converted to the reference current (`REFERENCE_CURRENT`, otherwise the largest of `LIMITS`) and the
deviation is related to F·U_Tp (`F_FACTOR`, `U_TP`): `assessment_relevance` "negligible" (< 10 %), "unchanged" (both
on the same side of F·U_Tp) or "relevant". Without these values of yours the relevance is not determined.
**LV network ↔ earthing measurement:** as a parallel branch Z_E,LV should also exceed Z_E of the earthing measurement
(`injection_comparison`). If it is smaller while the check against the fault test is satisfied, the earthing
measurement probably measures too much – a finding, not an error of the LV-network evaluation.

In [ ]:
COMPARISON = RATIO = None
RF = RF_EFF = LV_CHECK = LV_CHECK_EFF = pd.DataFrame()
if res is not None and not res.tables.get("station_frequency", pd.DataFrame()).empty:
    try:
        DB = earthing.Database(GROUNDMEAS_DB)          # opened read-only
    except (FileNotFoundError, ValueError) as ex:
        DB = None
        print("ℹ groundmeas database not available:", ex)
    if DB is not None:
        try:
            ENTRY = sites.load(CAMPAIGN_FOLDER).for_site(SITE)      # line of the sites file
        except ValueError as ex:
            ENTRY = None
            print("⚠", ex)
        LOCATION = GROUNDMEAS_LOCATION or (ENTRY.location if ENTRY is not None and ENTRY.location else None) or \
            earthing.location_from_site(SITE)
        CHOICE = DB.choose_location(LOCATION, exact_only=bool(GROUNDMEAS_LOCATION or (ENTRY is not None and ENTRY.location)),
                                    preset='GROUNDMEAS_LOCATION = "{}"')
        LOCATION_DB = CHOICE["location"]
        if CHOICE["note"]:
            print("ℹ", CHOICE["note"])
        if LOCATION_DB is None and len(CHOICE["candidates"]):
            display(Markdown("**Locations in groundmeas with similar names:**"))
            show(CHOICE["candidates"])
        if GROUNDMEAS_MEASUREMENTS:
            MEAS = DB.measurements()[lambda d: d["measurement_id"].isin(GROUNDMEAS_MEASUREMENTS)]
        else:
            MEAS = DB.measurements(location=LOCATION_DB) if LOCATION_DB else DB.measurements().iloc[0:0]
        MEAS = earthing.exclude_own(MEAS, CAMPAIGN, MEASUREMENT)   # do not compare the own entry with itself
        display(Markdown("**groundmeas: %s** – %d measurement(s) at the location “%s” (searched: “%s”)"
                         % (DB.path, len(MEAS), LOCATION_DB or "–", LOCATION)))
        show(MEAS)
        RESPONSE = earthing.frequency_response(DB, MEAS["measurement_id"])
        COMPARISON, RATIO = earthing.compare(res.table("station_frequency"), RESPONSE, measurement=MEASUREMENT)
        if len(RATIO):
            display(Markdown("**Ratio fault test / earthing measurement per frequency** (column `distance_match`: \"different\" = probes not comparable, not to be read as a reduction factor)"))
            show(RATIO)
        if COMPARISON["source"].isin([earthing.SOURCE_INJECTION, earthing.SOURCE_GROUNDMEAS_FAULT]).any():
            interactive.earthing_comparison(COMPARISON, RATIO).show()
        else:
            print("No matching values in groundmeas for “%s” – check the location/ids in section 1." % LOCATION)
        RF, RF_EFF = earthing.reduction_factor(res, RESPONSE, reference_current_A=REFERENCE_CURRENT,
                                               r_typical=R_TYPICAL)      # U_Tp, F and LIMITS from SETTINGS
        if len(RF_EFF) and R_TYPICAL is None:
            print("ℹ No classification of r at 50 Hz – R_TYPICAL is not set (section 1).")
        NOT_SET = [v for v, c in (("REFERENCE_CURRENT or LIMITS", "reference_current_A"), ("U_TP", "U_Tp_V"),
                                  ("F_FACTOR", "F_factor")) if len(RF_EFF) and RF_EFF[c].isna().all()]
        if NOT_SET:                    # relevance for the assessment: reference current, U_Tp and F of section 1
            print("ℹ Relevance of the deviation for the assessment not determined – not set (section 1): %s"
                  % ", ".join(NOT_SET))
        if len(RF_EFF):
            display(Markdown("**Reduction factor** – RMS value (`r_eff`, for comparison from the measured RMS values and from the shield current; classification at 50 Hz, trend over f, relevance for the assessment) and per harmonic (`r`); first rows: probe maximum"))
            show(RF_EFF); show(RF)
            interactive.reduction_factor(RF, RF_EFF).show()
        LV_CHECK, LV_CHECK_EFF = earthing.check_lv_network(res, RESPONSE)
        if len(LV_CHECK):
            display(Markdown("**LV network ↔ earthing measurement** (ratio Z_E,LV/Z_E ≥ 1 expected; parallel check: against the fault test)"))
            show(LV_CHECK_EFF); show(LV_CHECK)
            interactive.lv_network(res.table("station_lv_network"), LV_CHECK).show()
        for h in earthing.reduction_hints(RF_EFF, LV_CHECK):
            print("⚠", h)
        DB.close()

## 5 · Signal explorer – check the values at the curve

All channels of the recording: **at the top** the RMS value per 200-ms window over the whole file – on hover all
selected channels show RMS value, harmonics (H1, H3, H5, H7), THD and DC component at the same instant; **below**
instantaneous values, spectrum and characteristic values of the excerpt marked grey. Operation: select channels on the
left (Ctrl/⌘ or Shift for several), move the excerpt with the slider, with *instant* (fault inception, max window, …)
or ◀ ▶, set *width* and *up to order*. The max window gives the same values as the evaluation. The explorer also works
before the intervals are entered (then without the max window).

In [ ]:
EX = explorer.explorer(rec, res, alignment=ALIGNMENT) if SIGNAL_EXPLORER else None
EX

## 6 · Checks (optional)

In [ ]:
FT = VC = None
if res is not None:
    interactive.frequency_diagnosis(rec, res).show()
    P_RMS = CAMPAIGN_FOLDER / RMS_FILE if RMS_FILE else None
    if P_RMS is not None and not P_RMS.is_file():
        CANDIDATES = sorted(str(p.relative_to(CAMPAIGN_FOLDER)) for p in P_RMS.parent.glob("*.cfg")) if P_RMS.parent.is_dir() else []
        print("⚠ RMS_FILE not found: %s – check skipped.%s" % (
            RMS_FILE, (" Files in the folder: " + ", ".join(CANDIDATES)) if CANDIDATES else ""))
    elif P_RMS is not None:
        rec_rms = gf.open_recording(str(P_RMS))
        CORR.apply(rec_rms)                 # the same corrections for the RMS file
        for h in CORR.hints:                # hints on the RMS file (e.g. no sign there)
            print("⚠", h)
        display(Markdown("**Own true RMS value ↔ RMS value of the recorder** – deviations come mainly from DC offsets of the clamps (the recorder includes the DC, the evaluation removes it)."))
        show(rms.compare_with_instantaneous(rec, rec_rms, STEADY_STATE))
    if V_CURVE and res.roles.has("i_res"):
        FT = rms.fundamental_trend(rec, STEADY_STATE, res.roles)
        VC = rms.v_curve(FT["t_s"], FT["IH1_A"])
        print(i18n.format_text("Resonance curve: smallest fundamental current {:.2f} A at {:.2f} s{} {}",
                               VC.I_min_measured, VC.t_min_s, "" if np.isnan(VC.I_min_fit) else i18n.format_text(
                                   " (fitted parabola: {:.2f} A)", VC.I_min_fit), VC.note))
        show(FT)
        report.v_curve(FT["t_s"], FT["IH1_A"], VC, quantity="I_G")

## 7 · Export for the campaign evaluation and the report

* `<results>/<measurement>/<measurement>_result.json` – complete result (input of `evaluate_campaign.ipynb`)
* `<results>/<measurement>/tables/` – all tables as CSV, Markdown, one Excel workbook and one Word file with
  formatted tables
* `<figures>/<measurement>/` – figures in final size (15.9 cm wide, without titles) as PDF, SVG and PNG
* `<figures>/<measurement>/<measurement>_explorer.html` – signal explorer to pass on (opens in a browser without
  Python; for stations with the figures per frequency and the comparison with the earthing measurement)

In [ ]:
if res is not None:
    RESULTS = CAMPAIGN_FOLDER / LAYOUT.results / MEASUREMENT
    FIGURES = CAMPAIGN_FOLDER / LAYOUT.figures / MEASUREMENT
    p = res.save_json(LAYOUT.result_path(CAMPAIGN_FOLDER, MEASUREMENT))
    has_comparison = COMPARISON is not None and COMPARISON["source"].isin(
        [earthing.SOURCE_INJECTION, earthing.SOURCE_GROUNDMEAS_FAULT]).any()
    TABLES = dict(res.tables)
    if has_comparison:                 # comparison with groundmeas in the same workbook/Word file
        TABLES.update(earthing_comparison=COMPARISON, earthing_ratio=RATIO)
    TABLES.update({n: df for n, df in (("reduction_factor", RF), ("reduction_factor_eff", RF_EFF),
                                       ("lv_network_check", LV_CHECK), ("lv_network_check_eff", LV_CHECK_EFF)) if len(df)})
    tables = report.export_tables(TABLES, str(RESULTS / ("tables" if LAYOUT.language == "en" else "tabellen")),
                                  prefix=MEASUREMENT + "_", formats=("csv", "md"))
    figures = report.all_figures(res, str(FIGURES), rec=rec, prefix=MEASUREMENT + "_", formats=EXPORT_FORMATS)
    if VC is not None:
        figures += report.save(report.v_curve(FT["t_s"], FT["IH1_A"], VC, quantity="I_G"),
                               str(FIGURES / (MEASUREMENT + "_v_curve")), EXPORT_FORMATS, close=True)
    extra = {}
    if not res.tables.get("station_frequency", pd.DataFrame()).empty:
        extra["Earth potential rise and touch voltage per frequency"] = interactive.station_frequency(res)
    if has_comparison:
        figures += report.save(report.earthing_comparison(COMPARISON, RATIO),
                               str(FIGURES / (MEASUREMENT + "_earthing_comparison")), EXPORT_FORMATS, close=True)
        extra["Earthing measurement (groundmeas) and fault test per frequency"] = interactive.earthing_comparison(COMPARISON, RATIO)
    if len(RF_EFF):
        figures += report.save(report.reduction_factor(RF, RF_EFF), str(FIGURES / (MEASUREMENT + "_reduction_factor")),
                               EXPORT_FORMATS, close=True)
        extra["Reduction factor per harmonic"] = interactive.reduction_factor(RF, RF_EFF)
    if not res.tables.get("station_lv_network", pd.DataFrame()).empty:
        if len(LV_CHECK):              # with the earthing measurement (otherwise the standard figure station_lv_network)
            figures += report.save(report.lv_network(res.table("station_lv_network"), LV_CHECK),
                                   str(FIGURES / (MEASUREMENT + "_lv_network_injection")), EXPORT_FORMATS, close=True)
        extra["LV network: current and earthing impedance"] = interactive.lv_network(
            res.table("station_lv_network"), LV_CHECK if len(LV_CHECK) else None)
    if EXPLORER_HTML:
        h = explorer.explorer_html(rec, str(FIGURES / (MEASUREMENT + "_explorer.html")), res, figures=extra,
                                   alignment=ALIGNMENT)
        print("Signal explorer (HTML):", Path(h).relative_to(WORK_FOLDER))
    if RECORD_INTERVALS and DEMO:      # the demo changes nothing outside its campaign folder
        print("Register of the work folder left unchanged (demo)")
    elif RECORD_INTERVALS:
        args = (str(WORK_FOLDER), str(CFG.relative_to(WORK_FOLDER)), STEADY_STATE, PRE_FAULT, POST_FAULT, Z_FAULT)
        ok = register.record_intervals(*args)
        if not ok:                     # new file → read the register again (manual columns are kept)
            register.write_register(str(WORK_FOLDER))
            ok = register.record_intervals(*args)
        print("Register updated" if ok else "File not in the register – check the path")
    print("Result:", Path(p).relative_to(WORK_FOLDER))
    print("%d table files, %d figure files" % (len(tables), len(figures)))

### groundmeas – enter this measurement

The evaluated measurement can be entered as a **staged fault test** (`staged_fault_test`) in the groundmeas database –
its dashboard then shows it at the same location as the earthing measurement. **Station:** fault current, earth
potential rise and touch voltage per probe (with distance) and frequency – fault-related part, angle relative to I_F
– and shield current; **substation:** fault current per order, fault resistance = additional impedance. Location,
coordinates, asset type, voltage level and distance to the substation come from the sites file of the campaign
(created pre-filled from groundmeas the first time – **enter coordinates for a new location**). At first only the
planned entry is shown (dry run); it is written with `GROUNDMEAS_EXPORT = True` – after a backup copy of the database,
in one transaction, without changing existing entries. An entry made before is recognised;
`GROUNDMEAS_EXPORT = "replace"` renews the own entry.

In [ ]:
legacy.check_option("GROUNDMEAS_EXPORT", GROUNDMEAS_EXPORT, (False, True, "replace"))
PLAN = None
if res is not None:
    try:
        PLAN = earthing.export_to_groundmeas(
            res, CAMPAIGN_FOLDER, GROUNDMEAS_DB, location=GROUNDMEAS_LOCATION, write=bool(GROUNDMEAS_EXPORT),
            existing="replace" if GROUNDMEAS_EXPORT == "replace" else "skip", uncertain_mode=GROUNDMEAS_UNCERTAIN,
            description=GROUNDMEAS_DESCRIPTION, timezone=GROUNDMEAS_TIMEZONE, timestamp=GROUNDMEAS_TIMESTAMP,
            units=GROUNDMEAS_UNITS)
    except (FileNotFoundError, ValueError) as ex:
        print("ℹ groundmeas entry not possible:", ex)
if PLAN is not None:
    for h in PLAN["hints"]:
        print("ℹ", h)
    for w in PLAN["warnings"]:
        print("⚠ check before entering:", w)
    show(earthing.plan_overview({MEASUREMENT: PLAN}))
    if PLAN["written"]:
        print("✓ %s in groundmeas: measurement %d at the location “%s” (location %d) – backup copy: %s – now set GROUNDMEAS_EXPORT back to False."
              % ("renewed" if PLAN.get("renewed") else "entered", PLAN["measurement_id"], PLAN["location"],
                 PLAN["location_id"], PLAN["backup"]))
    elif not PLAN["writable"]:
        print("ℹ Not entered: %s" % ("; ".join(PLAN["reasons"]) or PLAN.get("reason", "")))
    else:
        display(Markdown("**Dry run** – the measurement would be entered like this (write with `GROUNDMEAS_EXPORT = True`):"))
    print("Description:", PLAN["measurement"]["description"])
    show(earthing.plan_items(PLAN))

---
**Conventions**
* Phasors are RMS phasors (cosine convention). Angles in the tables "ref. U_L1-L2" refer to the fundamental of the
  line-to-line voltage U_L1-L2 (order m: X_m·e^(−j·m·φ)) and are therefore independent of the time origin of the
  recording.
* Polarity: the active component of the fundamental must be positive (Iw > 0). If it is negative, the fault current
  is rotated by 180° (note in the table). With this orientation **Iv > 0 means over-compensation** (inductive excess).
* DC offset: removed per window by the median. RMS value = true RMS value over all frequencies, not only over the
  selected harmonics.